# FHIR assessment end-to-end validation

Run after the resource ingestion, Silver, and Gold activities complete. This notebook reads tables and Raw paths only; it does not modify data. It checks required Bronze metadata, current Silver version uniqueness and validity intervals, Gold keys, audit/watermark presence, and Raw response/sidecar counts.

This is validation evidence, not a substitute for successful pipeline runs on separate days or a demonstration of actual SCD Type 2 change history.

In [ ]:
RAISE_ON_FAILURE = False

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")

def table_exists(name):
    return spark.catalog.tableExists(name)

results = []
details = []

for resource_type in RESOURCE_TYPES:
    name = resource_type
    bronze_table = f"bronze.{name}"
    silver_table = f"silver.{name}"
    gold_table = f"gold.{name}"

    required_tables_present = all(table_exists(t) for t in [bronze_table, silver_table, gold_table])
    if not required_tables_present:
        missing = [t for t in [bronze_table, silver_table, gold_table] if not table_exists(t)]
        results.append({
            "resource_type": resource_type,
            "status": "FAIL",
            "issue": "Missing tables: " + ", ".join(missing)
        })
        continue

    bronze = spark.table(bronze_table)
    silver = spark.table(silver_table)
    gold = spark.table(gold_table)

    bronze_rows = bronze.count()
    bronze_distinct_ids = bronze.select("resource_id").distinct().count()
    required_metadata_fields = [
        "resource_id", "resource_json", "content_hash",
        "extraction_timestamp", "api_url_or_params", "saved_timestamp"
    ]
    missing_required_metadata = 0
    for field in required_metadata_fields:
        missing_required_metadata += bronze.filter(F.col(field).isNull()).count()

    duplicate_bronze_versions = (
        bronze.groupBy("resource_type", "resource_id", "content_hash")
        .count().filter(F.col("count") > 1).count()
    )

    silver_rows = silver.count()
    silver_ids = silver.select("resource_id").distinct().count()
    current_id_counts = (
        silver.groupBy("resource_id")
        .agg(F.sum(F.col("is_current").cast("int")).alias("current_versions"))
    )
    invalid_current_ids = current_id_counts.filter(F.col("current_versions") != 1).count()
    invalid_intervals = silver.filter(
        F.col("valid_to").isNotNull()
        & (F.col("valid_from").isNull() | (F.col("valid_from") >= F.col("valid_to")))
    ).count()
    multiple_version_ids = (
        silver.groupBy("resource_id").count().filter(F.col("count") > 1).count()
    )

    gold_key = {
        "Patient": "patient_id",
        "Encounter": "encounter_id",
        "Observation": "observation_id",
        "Condition": "condition_id"
    }[resource_type]
    gold_rows = gold.count()
    gold_missing_ids = gold.filter(F.col(gold_key).isNull()).count()
    gold_duplicate_ids = (
        gold.groupBy(gold_key).count()
        .filter(F.col(gold_key).isNotNull() & (F.col("count") > 1)).count()
    )

    resource_raw_dir = RAW_ROOT / f"resource={resource_type}"
    if resource_raw_dir.exists():
        raw_payloads = [
            p for p in resource_raw_dir.glob("extraction_date=*/run_id=*/page_*.json")
            if not p.name.endswith(".metadata.json")
        ]
        raw_sidecars = list(resource_raw_dir.glob("extraction_date=*/run_id=*/page_*.metadata.json"))
    else:
        raw_payloads = []
        raw_sidecars = []

    pass_checks = (
        missing_required_metadata == 0
        and duplicate_bronze_versions == 0
        and invalid_current_ids == 0
        and invalid_intervals == 0
        and gold_missing_ids == 0
        and gold_duplicate_ids == 0
        and len(raw_payloads) > 0
        and len(raw_payloads) == len(raw_sidecars)
    )
    status = "PASS" if pass_checks else "FAIL"

    details.append({
        "resource_type": resource_type,
        "bronze_rows": bronze_rows,
        "bronze_distinct_ids": bronze_distinct_ids,
        "bronze_required_nulls": missing_required_metadata,
        "bronze_duplicate_versions": duplicate_bronze_versions,
        "silver_rows": silver_rows,
        "silver_distinct_ids": silver_ids,
        "silver_invalid_current_ids": invalid_current_ids,
        "silver_invalid_intervals": invalid_intervals,
        "silver_ids_with_multiple_versions": multiple_version_ids,
        "gold_rows": gold_rows,
        "gold_missing_ids": gold_missing_ids,
        "gold_duplicate_ids": gold_duplicate_ids,
        "raw_payload_pages": len(raw_payloads),
        "raw_metadata_sidecars": len(raw_sidecars),
        "status": status
    })
    results.append({"resource_type": resource_type, "status": status, "issue": None if pass_checks else "Review the detail table."})

print("=== Per-resource validation ===")
if details:
    display(spark.createDataFrame(details))
else:
    print("No complete layer sets were found; inspect the missing-table results above.")

print("=== Overall validation ===")
display(spark.createDataFrame(results))

control_results = []
for control_table in ["dbo.fhir_ingestion_audit", "dbo.fhir_ingestion_watermarks"]:
    control_results.append({
        "table": control_table,
        "exists": table_exists(control_table),
        "rows": spark.table(control_table).count() if table_exists(control_table) else 0
    })
display(spark.createDataFrame(control_results))

if table_exists("dbo.fhir_ingestion_audit"):
    print("=== Successful audit runs by resource ===")
    display(
        spark.table("dbo.fhir_ingestion_audit")
        .filter(F.col("run_status") == "SUCCESS")
        .groupBy("resource_type", "run_id")
        .agg(
            F.count("*").alias("audit_rows"),
            F.sum(F.when(F.col("http_status") == 200, 1).otherwise(0)).alias("http_200_rows"),
            F.min("api_call_timestamp").alias("first_api_call_utc"),
            F.max("saved_timestamp").alias("last_saved_utc")
        )
        .orderBy("resource_type", "run_id")
    )

overall_pass = all(r["status"] == "PASS" for r in results) and all(r["exists"] and r["rows"] > 0 for r in control_results)
print("OVERALL:", "PASS" if overall_pass else "FAIL")
if RAISE_ON_FAILURE and not overall_pass:
    raise RuntimeError("FHIR assessment validation failed. Review the result tables above.")
